# Exploración CLR y PCA: taxonomía y funciones

Cuaderno canónico para los resultados finales de ordenación. Reúne el análisis
por género de MetaPhlAn y el análisis funcional de HUMAnN con una única
implementación de filtrado, CLR, PCA y biplot. Los experimentos y variantes
anteriores permanecen en `notebooks_archive/`.

## Resultados finales históricos

Las figuras que se eligieron de las versiones históricas se conservaron en
`results/figures/02_clr_pca_exploration/`. Al ejecutar este cuaderno se
regeneran junto con sus tablas de coordenadas y cargas.

![Biplot taxonómico final](../results/figures/02_clr_pca_exploration/taxonomic_genus_biplot_final.png)

![Biplot funcional final](../results/figures/02_clr_pca_exploration/functional_pathways_biplot_final.png)


## Configuración

Se eliminan las muestras atípicas usadas en los análisis originales. Las
características presentes en menos de 5% de las muestras se agregan como
`Other`; esto evita descartes silenciosos y conserva la masa de abundancia.
El CLR se aplica una única vez, con pseudoconteo, y no se estandariza de nuevo
antes de PCA.


In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.ordination import (
    load_functional_pathways,
    load_metaphlan_genus,
    plot_clr_biplot,
    prepare_composition,
    run_clr_pca,
)
from src.paths import DATASETS_DIR, analysis_output_dirs

ANALYSIS = "02_clr_pca_exploration"
OUTPUTS = analysis_output_dirs(ANALYSIS)
OUTLIERS = (
    "37082_3#16", "37082_2#4", "37035_2#22",
    "37035_7#18", "36703_3#20", "36703_3#4",
)
PREVALENCE = 0.05
metadata = pd.read_excel(DATASETS_DIR / "metadata_LATINBIOTA_MEXICO.xlsx", sheet_name="Data")


## Funciones compartidas de análisis y exportación


In [3]:
def analyse_and_export(name: str, abundance: pd.DataFrame, title: str) -> dict:
    # Ejecuta el flujo final CLR/PCA y escribe las salidas reproducibles.
    composition, aligned_metadata = prepare_composition(
        abundance,
        metadata,
        excluded_samples=OUTLIERS,
        prevalence=PREVALENCE,
    )
    result = run_clr_pca(composition)
    scores = result["scores"].join(aligned_metadata[["Lifestyle"]])
    scores.to_csv(OUTPUTS["tables"] / f"{name}_pca_scores.csv")
    result["loadings"].to_csv(OUTPUTS["tables"] / f"{name}_pca_loadings.csv")

    fig, _ = plot_clr_biplot(result, aligned_metadata, title=title)
    fig.savefig(OUTPUTS["figures"] / f"{name}_clr_pca_biplot.png", dpi=300, bbox_inches="tight")
    fig.savefig(OUTPUTS["figures"] / f"{name}_clr_pca_biplot.pdf", bbox_inches="tight")
    plt.show()

    explained = result["pca"].explained_variance_ratio_[:2].sum()
    print(f"{name}: {composition.shape[0]} muestras, {composition.shape[1]} características; PC1+PC2 = {explained:.1%}")
    return {**result, "composition": composition, "metadata": aligned_metadata, "scores": scores}


## Resultado final 1 — composición taxonómica por género


In [4]:
genus_abundance = load_metaphlan_genus(DATASETS_DIR / "latinbiota_merge_metaphlan_data.csv")
genus_result = analyse_and_export(
    "taxonomic_genus",
    genus_abundance,
    "PCA con CLR: composición taxonómica por género",
)
genus_result["loadings"].head(10)


IndexError: boolean index did not match indexed array along axis 0; size of axis is 1263 but size of corresponding boolean axis is 1262

## Resultado final 2 — rutas funcionales no estratificadas


In [ ]:
functional_abundance = load_functional_pathways(
    DATASETS_DIR / "latinbiota_pathabundance_unstratified_relab.tsv"
)
functional_result = analyse_and_export(
    "functional_pathways",
    functional_abundance,
    "PCA con CLR: rutas funcionales",
)
functional_result["loadings"].head(10)


## Entregables finales


In [ ]:
pd.DataFrame(
    {
        "analysis": ["taxonomic_genus", "functional_pathways"],
        "samples": [len(genus_result["scores"]), len(functional_result["scores"])],
        "features_after_filter": [genus_result["composition"].shape[1], functional_result["composition"].shape[1]],
        "variance_pc1_pc2": [
            genus_result["pca"].explained_variance_ratio_[:2].sum(),
            functional_result["pca"].explained_variance_ratio_[:2].sum(),
        ],
    }
).set_index("analysis")
